**`03_prepare_administrative_units`**

Prepare initial layer of administrative units with geometries and `AdminId` identifiers.

# Administrative units identifiers
``openplaces`` organizes its data by ``admin_id`` (data class: ``AdminId``).

``admin_id`` is a geographical administrative index with hierarchical ``.levels`` of any depth.

- **0** - countries
- **1** - states/departments/...
- **2** - counties/municipalities/...
- **3** - subdivisions/towns/...
- **4** - neighborhoods/...
- **5** - ...

The initial built is derived from ISO and the Global Administrative Database (GADM).

In [ ]:
from openplaces.api import get_admin0, get_admin1, get_admin2
from openplaces.io.admin import get_admin0_iso, get_admin1_iso
from openplaces.io.ingester import Ingester
from openplaces.recipe import get_recipe_by_id
from openplaces.timing import get_timer
from openplaces.utils import pretty_print

In [ ]:
# Redownload and reprocess source files? (if downloaded files exist)
REDOWNLOAD = False

# Reprocess source data file? (if output files exist)
REPROCESS = False

In [ ]:
# Start timer
# timer = get_timer('prepare_administrative_units', verbose=True)

# `admin0`: countries / territories
The highest level of the administrative hierarchy.
## ISO countries
Top-level administrative identifiers, gap-filled to match GADM, ships with `openplaces`

In [ ]:
get_admin0_iso().sample(5).sort_index()

## GADM level 0
Example of how to download and ingest data from the Internet using a `recipe`:

In [ ]:
pretty_print(get_recipe_by_id('admin-gadm-4~1_admin0'))

In [ ]:
ingester = Ingester('admin-gadm-4~1_admin0', verbose=True)
ingester.ingest(reprocess=REPROCESS, redownload=REDOWNLOAD)

### Read result

In [ ]:
admin0 = get_admin0()
admin0.sample(5).sort_index()

# ``admin1``: states / departments

## ISO states

In [ ]:
admin1_iso = get_admin1_iso()
admin1_iso.sample(5).sort_index()

## GADM level 1

In [ ]:
pretty_print(get_recipe_by_id('admin-gadm-4~1_admin1'))

In [ ]:
ingester = Ingester('admin-gadm-4~1_admin1', verbose=True)
ingester.ingest(reprocess=REPROCESS)

In [ ]:
admin1 = get_admin1()
admin1.sample(5).sort_index()

# ``admin2``: counties / municipalities

## GADM level 2

In [ ]:
pretty_print(get_recipe_by_id('admin-gadm-4~1_admin2'))

In [ ]:
ingester = Ingester('admin-gadm-4~1_admin2', verbose=True)
ingester.ingest(reprocess=REPROCESS)

In [ ]:
admin2 = get_admin2(
    columns=['name', 'type', 'admin1_name', 'admin0_name', 'admin2_id_gadm']
)
admin2.sample(5).sort_index()

# ``admin3``: towns / ...

## GADM level 3

In [ ]:
pretty_print(get_recipe_by_id('admin-gadm-4~1_admin3'))

In [ ]:
ingester = Ingester('admin-gadm-4~1_admin3', verbose=True)
ingester.ingest(reprocess=REPROCESS)

### Issue 1: Linkage to Admin-2 not fully successful

In [ ]:
import pandas as pd

from openplaces.path import cache_path

admin3_recipe = get_recipe_by_id('admin-gadm-4~1_admin3')
admin3_path = cache_path(
    admin3_recipe['admin_id'],
    admin3_recipe['entity'],
    filename=admin3_recipe['cache_filename'],
)
admin3 = pd.read_parquet(admin3_path)

admin3 = admin3.join(
    admin2.reset_index()
    .set_index('admin2_id_gadm')[['admin2_id', 'name']]
    .rename(columns={'name': 'admin2_name_test'}),
    on='admin2_id_gadm',
)
admin3[
    admin3['admin2_name_test'].ne(admin3['admin2_name'])
].sample(5).T

### Issue 2: GADM does not fully cover the globe

In [ ]:
import geopandas as gpd

admin3_geo_path = cache_path(
    admin3_recipe['admin_id'],
    admin3_recipe['entity'],
    filename=admin3_recipe['cache_filename'] + '_geo',
)
admin3_geo = gpd.read_parquet(admin3_geo_path)
admin3_geo.plot()